# UMAP visualization of AlphaEarth and Sentinel-2 embeddings

**Phase 2 task 2** for the RSE paper. Generates the visual evidence for the geographic-shortcut diagnosis (Section 5.8.3 of the manuscript).

**Output figures (saved to `Manuscript/Paper_Alpha_Earth/exp6_borrowing_strategies/figures/main/`):**
- `fig11_umap_ae_by_country.{pdf,png}`
- `fig11_umap_ae_by_climate.{pdf,png}`
- `fig11_umap_ae_by_crop.{pdf,png}`
- `fig11_umap_s2_by_country.{pdf,png}`
- `fig11_umap_s2_by_climate.{pdf,png}`
- `fig11_umap_s2_by_crop.{pdf,png}`
- `fig11_umap_ae_vs_s2_grid.{pdf,png}` ← used in the paper

**Estimated runtime on Colab (free tier, CPU):** ~10-15 min for 5,000 parcels per country × 11 countries × 2 features.

## 1. Mount Google Drive and locate the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

# Try to auto-locate the Alpha_Crops folder. If your shortcut is at a different
# location, edit PROJECT_ROOT manually.
candidates = [
    Path('/content/drive/MyDrive/Alpha_Crops'),
    Path('/content/drive/Shareddrives/Alpha_Crops'),
    Path('/content/drive/MyDrive/Research/Alpha_Crops'),
]
PROJECT_ROOT = None
for c in candidates:
    if c.exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Alpha_Crops folder not found. Edit PROJECT_ROOT to the correct path.\n'
        'Tried: ' + '\n'.join(str(c) for c in candidates))
print(f'Project root: {PROJECT_ROOT}')

FEATURES_DIR = PROJECT_ROOT / 'data' / 'eu_processed' / 'features'
FIG_DIR = PROJECT_ROOT / 'Manuscript' / 'Paper_Alpha_Earth' / 'exp6_borrowing_strategies' / 'figures' / 'main'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Features: {FEATURES_DIR}')
print(f'Output figures: {FIG_DIR}')

## 2. Install dependencies

In [ ]:
!pip install -q umap-learn pyarrow

## 3. Load and concatenate features (train + val + test)

All splits are combined since UMAP is a per-country visualization, not a model evaluation.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 11 countries used in the paper's main analysis (2021 LPIS aligned)
MAIN_COUNTRIES_2021 = ['AT', 'BE', 'DE', 'EE', 'LT', 'LV', 'PT', 'RO', 'SE', 'SI', 'SK']

# Strict 2021 alignment requires excluding DE_BB (which uses 2023 LPIS),
# matching the paper's main-analysis methodology (Section 3.1).
EXCLUDE_PARCEL_PREFIXES = ('DE_BB_',)

CLIMATE_ZONE = {
    'AT': 'Continental', 'BE': 'Atlantic', 'DE': 'Atlantic',
    'EE': 'Nordic-Baltic', 'LT': 'Nordic-Baltic', 'LV': 'Nordic-Baltic',
    'PT': 'Mediterranean', 'RO': 'Southeast', 'SE': 'Nordic-Baltic',
    'SI': 'Mediterranean', 'SK': 'Continental',
}


def load_features(feature_set):
    """Concatenate train+val+test for a given feature_set ('ae' or 's2').

    Filters to the 11 paper countries AND drops DE_BB parcels (2023 LPIS,
    not aligned with the strict 2021 main analysis).
    """
    parts = []
    for split in ['train', 'val', 'test']:
        f = FEATURES_DIR / feature_set / f'features_{split}.parquet'
        if f.exists():
            parts.append(pd.read_parquet(f))
    df = pd.concat(parts, ignore_index=True)
    df = df[df['country'].isin(MAIN_COUNTRIES_2021)].copy()

    # Strict 2021 alignment: drop DE_BB parcels (uses 2023 LPIS)
    pid = df['parcel_id'].astype(str)
    bb_mask = pid.str.startswith(EXCLUDE_PARCEL_PREFIXES)
    n_dropped = int(bb_mask.sum())
    if n_dropped > 0:
        df = df[~bb_mask.values].copy()
        print(f'  [strict 2021] dropped {n_dropped:,} DE_BB parcels (2023 LPIS)')

    df['climate_zone'] = df['country'].map(CLIMATE_ZONE)
    return df


df_ae = load_features('ae')
df_s2 = load_features('s2')
print(f'AE shape: {df_ae.shape}')
print(f'S2 shape: {df_s2.shape}')
print(f'Per-country counts (AE) — DE should be ~99,988 after BB exclusion:')
print(df_ae['country'].value_counts().sort_index())

## 4. Stratified subsample for UMAP (5,000 parcels per country)

UMAP scales poorly past ~50k samples; subsampling per country keeps within-country structure visible.

In [ ]:
import psutil, os
mem = psutil.Process(os.getpid()).memory_info().rss / 1e9
print(f'RAM used: {mem:.2f} GB')

SAMPLE_PER_COUNTRY = 5000
SEED = 42

def stratified_subsample(df, n_per_country, seed):
    parts = []
    rng = np.random.RandomState(seed)
    for country, grp in df.groupby('country', sort=False):
        if len(grp) > n_per_country:
            idx = rng.choice(grp.index, size=n_per_country, replace=False)
            parts.append(grp.loc[idx])
        else:
            parts.append(grp)
    return pd.concat(parts, ignore_index=True)

# 1) 先在 AE 上抽样（小得多）
df_ae_sub = stratified_subsample(df_ae, SAMPLE_PER_COUNTRY, SEED)
print(f'AE subsample: {df_ae_sub.shape}')

# 2) 立刻释放 df_ae 大对象（5万样本不再需要它）
del df_ae
import gc; gc.collect()

# 3) 用 merge 而不是 set_index().loc 做对齐——内存友好
df_s2_sub = df_ae_sub[['parcel_id']].merge(df_s2, on='parcel_id', how='inner')
print(f'S2 subsample: {df_s2_sub.shape}')

# 释放 df_s2 全量
del df_s2
gc.collect()

# 重新对齐顺序到 AE（小数据上 set_index 安全）
df_s2_sub = df_s2_sub.set_index('parcel_id').reindex(df_ae_sub['parcel_id']).reset_index()
assert (df_ae_sub['parcel_id'].values == df_s2_sub['parcel_id'].values).all()
print('parcel_id alignment verified ✓')

## 5. Run UMAP on AE and S2 features

We use only spectral / embedding features — explicitly excluding `longitude`, `latitude`, `log_area` so the geographic structure we observe is purely from the embedding space, not from coordinates.

In [ ]:
# 丢掉 ~1954 个 S2 没匹配上的行，让 AE / S2 严格对齐
nan_mask = df_s2_sub.isna().any(axis=1)
print(f'Dropping {nan_mask.sum()} rows where S2 features are NaN')

df_ae_sub = df_ae_sub.loc[~nan_mask.values].reset_index(drop=True)
df_s2_sub = df_s2_sub.loc[~nan_mask.values].reset_index(drop=True)

assert len(df_ae_sub) == len(df_s2_sub)
assert (df_ae_sub['parcel_id'].values == df_s2_sub['parcel_id'].values).all()
print(f'Final: AE={df_ae_sub.shape}, S2={df_s2_sub.shape}')

In [ ]:
from sklearn.preprocessing import StandardScaler
import umap

META_COLS = {
    'parcel_id', 'country', 'climate_zone', 'crop',
    'log_area', 'longitude', 'latitude', 'area_ha',
    'EC_hcat_n', 'split', 'size_class',
    'label_tier_a', 'label_tier_b', 'label_tier_c',
    'label_tier_a_encoded', 'label_tier_b_encoded', 'label_tier_c_encoded',
}

def run_umap(df, label_col='label_tier_b'):
    feat_cols = [c for c in df.columns if c not in META_COLS]
    X = df[feat_cols].values.astype(np.float32)
    print(f'  features: {X.shape}')
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, metric='euclidean',
                         random_state=SEED, n_jobs=1)
    return reducer.fit_transform(X_scaled)

print('=== AlphaEarth ===')
coords_ae = run_umap(df_ae_sub)
print('=== Sentinel-2 ===')
coords_s2 = run_umap(df_s2_sub)

df_ae_sub['umap_x'] = coords_ae[:, 0]
df_ae_sub['umap_y'] = coords_ae[:, 1]
df_s2_sub['umap_x'] = coords_s2[:, 0]
df_s2_sub['umap_y'] = coords_s2[:, 1]

# crop column from label_tier_b (textual labels)
if 'label_tier_b' in df_ae_sub.columns:
    df_ae_sub['crop'] = df_ae_sub['label_tier_b']
    df_s2_sub['crop'] = df_s2_sub['label_tier_b']
print('Done.')

## 6. Plot 6 individual panels and the combined 3×2 grid

In [ ]:
import matplotlib.pyplot as plt

CLIMATE_PALETTE = {
    'Atlantic':      '#1f77b4',
    'Continental':   '#2ca02c',
    'Mediterranean': '#d62728',
    'Nordic-Baltic': '#9467bd',
    'Southeast':     '#ff7f0e',
}

CROP_PALETTE = {
    'grassland':         '#2ca02c',
    'winter_cereals':    '#bcbd22',
    'maize':             '#ff7f0e',
    'potatoes':          '#8c564b',
    'root_crops':        '#e377c2',
    'orchards':          '#9467bd',
    'vegetables':        '#17becf',
    'strawberries':      '#d62728',
    'tree_wood_forest':  '#1f77b4',
    'other_crops':       '#7f7f7f',
}

def plot_panel(coords, labels, palette, title, ax, point_size=1.5):
    for lab in sorted(set(palette.keys()) & set(labels.unique())):
        mask = labels.values == lab
        ax.scatter(coords[mask, 0], coords[mask, 1], c=palette[lab],
                    s=point_size, alpha=0.55, label=str(lab), edgecolors='none')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

In [ ]:
# Country palette (11 countries, distinct colors)
country_palette = {c: plt.cm.tab20(i / len(MAIN_COUNTRIES_2021))
                    for i, c in enumerate(sorted(MAIN_COUNTRIES_2021))}

# 6 individual figures
for feat_name, df in [('ae', df_ae_sub), ('s2', df_s2_sub)]:
    coords = df[['umap_x', 'umap_y']].values
    for color_by, palette, title_suffix in [
        ('country', country_palette, 'country'),
        ('climate_zone', CLIMATE_PALETTE, 'climate zone'),
        ('crop', CROP_PALETTE, 'crop class'),
    ]:
        fig, ax = plt.subplots(figsize=(7, 7))
        plot_panel(coords, df[color_by], palette,
                   f'{feat_name.upper()} embeddings — coloured by {title_suffix}', ax)
        ax.legend(markerscale=4, fontsize=8, loc='upper right',
                  bbox_to_anchor=(1.25, 1.0))
        fig.tight_layout()
        out = FIG_DIR / f'fig11_umap_{feat_name}_by_{color_by}'
        fig.savefig(f'{out}.pdf', bbox_inches='tight')
        fig.savefig(f'{out}.png', bbox_inches='tight', dpi=200)
        plt.close(fig)
        print(f'saved {out}.pdf')

# Combined 3x2 grid (used in the paper)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for row_idx, (feat_name, df) in enumerate([('ae', df_ae_sub), ('s2', df_s2_sub)]):
    coords = df[['umap_x', 'umap_y']].values
    plot_panel(coords, df['country'], country_palette,
               f'{feat_name.upper()}: by country', axes[row_idx, 0])
    plot_panel(coords, df['climate_zone'], CLIMATE_PALETTE,
               f'{feat_name.upper()}: by climate zone', axes[row_idx, 1])
    plot_panel(coords, df['crop'], CROP_PALETTE,
               f'{feat_name.upper()}: by crop class', axes[row_idx, 2])

fig.suptitle('UMAP of feature embeddings — geographic vs semantic structure',
             fontsize=13, y=1.00)
fig.tight_layout()
out = FIG_DIR / 'fig11_umap_ae_vs_s2_grid'
fig.savefig(f'{out}.pdf', bbox_inches='tight')
fig.savefig(f'{out}.png', bbox_inches='tight', dpi=200)
print(f'\n*** Main paper figure saved: {out}.pdf ***')
plt.show()

## 7. Done

All 7 PDF + 7 PNG files are now in:
`Manuscript/Paper_Alpha_Earth/exp6_borrowing_strategies/figures/main/`

The main paper figure (Section 5.8.3) is `fig11_umap_ae_vs_s2_grid.pdf`. The LaTeX source already references this filename — re-compile and the figure will appear.

**Sanity check:** in the AE panels, country colors should form tight clusters (geographic shortcut visible); the crop-coloured AE panel should look more diffuse. In S2 panels, the country structure should be weaker and the crop structure relatively more visible. This is the visual counterpart of CKA<sub>crop</sub>=0.244 (AE) vs 0.056 (S2).